# 다이캐스팅 불량 유형 판별 모델 (Product Type 1)

ml_multilabel_type1

## 목표
1. 데이터를 불러오고(컬럼 구조 이해)
2. 불량(Defects)을 **표면/구조 그룹**으로 묶어 **Multi-label 타겟(y)**을 만든 뒤
3. 공정(Process) + 센서(Sensor) 변수로 **XGBoost 모델**을 학습합니다.
4. **데이터 누출(leakage)** 위험이 있는 컬럼을 자동으로 제거하여 **신뢰도 있는 모델**을 만듭니다.
5. Feature importance / (선택) SHAP / “불량 최소 조건” 탐색까지 이어집니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# XGBoost (설치가 안 되어 있으면 아래 주석을 해제 후 실행)
# !pip -q install xgboost
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

plt.rcParams['font.family'] = 'Malgun Gothic' 
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 로드

이 프로젝트에서는 `product_type_1.csv`를 사용합니다.

이 CSV는 **2줄 헤더(MultiIndex)** 구조일 수 있습니다.
- 상위 레벨: process / sensor / defects
- 하위 레벨: velocity_1, bubble_1 등

그래서 `header=[0,1]`로 먼저 읽어보고,
- MultiIndex면 그대로 사용
- 아니면 일반 컬럼(flat)으로 처리합니다.


In [ ]:

try:
    df1 = pd.read_csv("data_processed/product_type_1.csv", header=[0, 1])
    multi = True
except Exception:
    df1 = pd.read_csv("data_processed/product_type_1.csv", header=[0, 1])
    multi = False

print("MultiIndex header:", multi)
print("Shape:", df1.shape)

if multi:
    print("Top-level columns:", df1.columns.levels[0].tolist())
else:
    print("Columns sample:", df1.columns[:15].tolist())

df1.head()


## 2. process / sensor / defects 블록 분리

- MultiIndex라면 `df1['process']` 처럼 접근 가능
- flat 컬럼이라면 `process_...`, `sensor_...`, `defects_...` 접두어로 찾아 분리

아래 코드는 두 경우를 모두 안전하게 처리합니다.


In [ ]:
def is_multiindex_columns(df: pd.DataFrame) -> bool:
    return isinstance(df.columns, pd.MultiIndex)

def get_block(df: pd.DataFrame, block_name: str) -> pd.DataFrame:
    if is_multiindex_columns(df):
        return df[block_name].copy()
    prefix = block_name.lower() + "_"
    cols = [c for c in df.columns if str(c).lower().startswith(prefix)]
    return df[cols].copy()

process_df = get_block(df1, "process")
sensor_df  = get_block(df1, "sensor")
defects_df = get_block(df1, "defects")

print("process_df:", process_df.shape)
print("sensor_df :", sensor_df.shape)
print("defects_df:", defects_df.shape)
print("\n[defects columns]")
print(defects_df.columns.tolist())


## 3. defect 그룹화 → multi-label 타겟(y) 만들기

우리가 예측할 y는 2개입니다.

- **표면**: dent/stain/exfoliation 중 하나라도 있으면 1
- **구조**: short_shot/bubble/deformation 중 하나라도 있으면 1

주의:
- defects 값이 0/1 뿐 아니라 2,3도 있을 수 있으므로
  - **0이면 양품**
  - **0초과면 불량(1로 통일)**
  로 이진화합니다.


In [ ]:
defects_numeric = defects_df.apply(pd.to_numeric, errors="coerce").fillna(0)
defects_bin = (defects_numeric > 0).astype(int)

defect_groups = {
    "표면": [
        "exfoliation_1",
        "stain_1",
        "dent_1",
        "exfoliation_2"
    ],

    "구조": [
        "short_shot_1",
        "bubble_1",
        "short_shot_2",
        "bubble_2",
        "deformation_1",
        "deformation_2"
    ]
}

def normalize_defect_colname(col):
    s = str(col).lower()
    return s.replace("defects_", "") if s.startswith("defects_") else s

map_to_real = {normalize_defect_colname(c): c for c in defects_bin.columns}

y = pd.DataFrame(index=df1.index)
for group_name, cols in defect_groups.items():
    real_cols = [map_to_real[c] for c in cols if c in map_to_real]
    if len(real_cols) == 0:
        raise KeyError(f"[{group_name}] 그룹에 해당하는 defects 컬럼을 찾지 못했습니다: {cols}")
    y[group_name] = defects_bin[real_cols].any(axis=1).astype(int)

print("[표면/구조 불량 비율(%)]")
display((y.mean() * 100).round(2).to_frame("rate_%"))
y.head()


In [ ]:
# defect_groups에 정의된 컬럼 목록(정규화 기준)
group_defined_cols = set()
for cols in defect_groups.values():
    group_defined_cols.update(cols)

# 실제 defects 컬럼 목록(정규화 기준)
actual_defect_cols = set(map_to_real.keys())

# 그룹엔 있는데 실제엔 없는 컬럼
missing_in_data = sorted(group_defined_cols - actual_defect_cols)

# 실제엔 있는데 그룹에 없는 컬럼
not_grouped = sorted(actual_defect_cols - group_defined_cols)

print("\n[그룹 정의 컬럼 수]", len(group_defined_cols))
print("[실제 defects 컬럼 수]", len(actual_defect_cols))

print("\n[그룹에 정의했지만 실제 데이터엔 없는 컬럼]")
print(missing_in_data if missing_in_data else "없음")

print("\n[실제 defects 컬럼 중 그룹에 포함되지 않은 컬럼]")
print(not_grouped if not_grouped else "없음")

## 4. X 만들기 + 데이터 누출(leakage) 방지

X는 **process + sensor**만 사용합니다.

그리고 아래는 “미래 예측에서 알 수 없거나”, “식별자/시간”이라서  
모델이 **공정 원인이 아니라 꼼수 패턴을 외우게** 만들 수 있습니다.

- process_id: 단순 식별번호
- process_shot: 생산 순서/카운터(시간 변수 성격)
- process_product_type: 이번 분석에서는 product_type_1만 사용 → 불필요/혼입 위험
- defect_flag_is_defect / is_defect: 이름부터 정답 힌트 가능

그래서 이런 컬럼을 **자동 탐지해서 제거**합니다.


In [ ]:
X = pd.concat([process_df, sensor_df], axis=1).copy()
X = X.drop(columns=["id","shot"], errors="ignore")
print("X shape (before drop):", X.shape)

def col_to_str(c):
    if isinstance(c, tuple):
        return "_".join([str(x) for x in c]).lower()
    return str(c).lower()

leak_keywords = [
    "process_id",
    "process_shot",
    "process_product_type",
    "defect_flag_is_defect",
    "is_defect",
]

leak_cols = [c for c in X.columns if any(k in col_to_str(c) for k in leak_keywords)]
if leak_cols:
    print("drop leakage cols:", leak_cols)
    X = X.drop(columns=leak_cols, errors="ignore")
else:
    print("leakage 키워드 컬럼 없음")

defect_like = [c for c in X.columns if "defects" in col_to_str(c)]
if defect_like:
    print("drop defects-like cols from X:", defect_like)
    X = X.drop(columns=defect_like, errors="ignore")

print("X shape (after drop):", X.shape)


## 5. Train/Test 분리 (multi-label stratify)

y가 (표면, 구조) 2개 컬럼이므로 `stratify=y`를 그대로 쓰면 오류가 나거나 불안정할 수 있습니다.

그래서 (표면,구조) 조합을 문자열로 만들어 strata로 사용합니다.
- 00, 10, 01, 11 같은 형태


In [ ]:
strata = y.astype(str).agg("".join, axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strata
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)

print("\n[라벨 조합 분포(전체)]")
display(strata.value_counts(normalize=True).sort_index().to_frame("ratio"))


In [ ]:
# 1. train 기준 결측치 확인
# =========================
train_missing_count = X_train.isna().sum()
train_missing_ratio = X_train.isna().mean() * 100

missing_summary = pd.DataFrame({
    "missing_count": train_missing_count,
    "missing_ratio(%)": train_missing_ratio
}).sort_values("missing_ratio(%)", ascending=False)

print("[X_train 결측치 요약]")
display(missing_summary[missing_summary["missing_count"] > 0])

print(f"결측치가 있는 컬럼 수: {(train_missing_count > 0).sum()}")
print(f"전체 컬럼 수: {X_train.shape[1]}")

In [ ]:
# =========================
# 2. train의 median으로 결측치 대체
# =========================
numeric_cols = X_train.select_dtypes(include=[np.number]).columns

imputer = SimpleImputer(strategy="median")

X_train[numeric_cols] = imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = imputer.transform(X_test[numeric_cols])

print("[결측치 대체 후 확인]")
print("X_train 결측치 총합:", X_train.isna().sum().sum())
print("X_test 결측치 총합 :", X_test.isna().sum().sum())

In [ ]:
# ==============================
# Stage 1 : 불량 / 정상 라벨 생성
# ==============================

y_train_stage1 = (y_train.sum(axis=1) > 0).astype(int)
y_test_stage1  = (y_test.sum(axis=1) > 0).astype(int)

print("불량 비율 (train):")
print(y_train_stage1.mean())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd

# ==============================
# Stage 1 : 불량 / 정상 라벨
# ==============================
y_train_stage1 = (y_train.sum(axis=1) > 0).astype(int)
y_test_stage1  = (y_test.sum(axis=1) > 0).astype(int)

# ==============================
# Train → Train / Validation 분리
# ==============================
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train,
    y_train_stage1,
    test_size=0.2,
    random_state=42,
    stratify=y_train_stage1
)

print("Train:", X_train_sub.shape)
print("Validation:", X_val.shape)

# ==============================
# 기본 불균형 비율
# ==============================
neg = (y_train_sub == 0).sum()
pos = (y_train_sub == 1).sum()
base_weight = neg / pos

print("정상 개수:", neg)
print("불량 개수:", pos)
print("base_weight:", round(base_weight, 4))

# ==============================
# 튜닝 후보
# ==============================
weight_candidates = [base_weight, base_weight * 1.5, base_weight * 2]
threshold_candidates = [0.1, 0.15, 0.2, 0.25, 0.3]

param_candidates = [
    {
        "n_estimators": 400,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "gamma": 0
    },
    {
        "n_estimators": 600,
        "max_depth": 4,
        "learning_rate": 0.03,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "min_child_weight": 1,
        "gamma": 0
    },
    {
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 3,
        "gamma": 0.1
    }
]

# ==============================
# 전체 조합 탐색 (Validation 기준)
# ==============================
results = []
best_model = None
best_threshold = None
best_score = -1

exp_id = 0

for params in param_candidates:
    for w in weight_candidates:

        exp_id += 1
        print("=" * 70)
        print(f"[실험 {exp_id}] params={params}, scale_pos_weight={round(w,4)}")

        model = XGBClassifier(
            **params,
            scale_pos_weight=w,
            random_state=42,
            eval_metric="logloss",
            n_jobs=-1
        )

        # train으로 학습
        model.fit(X_train_sub, y_train_sub)

        # validation으로 평가
        y_prob = model.predict_proba(X_val)[:, 1]

        for th in threshold_candidates:

            y_pred = (y_prob >= th).astype(int)

            report = classification_report(
                y_val,
                y_pred,
                output_dict=True,
                zero_division=0
            )

            precision_1 = report["1"]["precision"]
            recall_1 = report["1"]["recall"]
            f1_1 = report["1"]["f1-score"]
            accuracy = report["accuracy"]

            results.append({
                "exp_id": exp_id,
                "threshold": th,
                "scale_pos_weight": w,
                "n_estimators": params["n_estimators"],
                "max_depth": params["max_depth"],
                "learning_rate": params["learning_rate"],
                "subsample": params["subsample"],
                "colsample_bytree": params["colsample_bytree"],
                "min_child_weight": params["min_child_weight"],
                "gamma": params["gamma"],
                "precision_1": precision_1,
                "recall_1": recall_1,
                "f1_1": f1_1,
                "accuracy": accuracy
            })

            score = (recall_1 * 0.7) + (f1_1 * 0.3)

            if score > best_score:
                best_score = score
                best_model = model
                best_threshold = th

results_df = pd.DataFrame(results).sort_values(
    ["recall_1", "f1_1", "precision_1"],
    ascending=False
)

print("\n[Validation 기준 상위 15개 결과]")
display(results_df.head(15))

print("\n[선택된 최고 조합]")
display(results_df.iloc[[0]])

# ==============================
# Test 데이터로 최종 평가
# ==============================
y_prob_best = best_model.predict_proba(X_test)[:, 1]
y_pred_best = (y_prob_best >= best_threshold).astype(int)

print(f"\n=== Stage 1 최종 Test 결과 (threshold={best_threshold}) ===")
print(classification_report(y_test_stage1, y_pred_best, zero_division=0))

제조 공정이라는게 불량을 놓쳐서는안된다.
32% 1차에서 불량 여부이라고 판정내린 데이터들을
2차 모델에서 정상 / 불량 유형(표면, 구조) 다시 한번 가설

In [ ]:
# !pip install shap

import shap
import matplotlib.pyplot as plt
import pandas as pd

# ==============================
# SHAP용 feature 이름 정리
# ==============================
X_train_shap = X_train.copy()
X_test_shap = X_test.copy()

# 혹시 컬럼명이 MultiIndex면 문자열로 변환
X_train_shap.columns = [str(c) for c in X_train_shap.columns]
X_test_shap.columns = [str(c) for c in X_test_shap.columns]

# ==============================
# SHAP Explainer 생성
# ==============================
explainer = shap.TreeExplainer(best_model)

# test 전체를 다 쓰면 느릴 수 있으니 일부 샘플만 사용
X_shap_sample = X_test_shap.sample(min(200, len(X_test_shap)), random_state=42)

shap_values = explainer.shap_values(X_shap_sample)

# ==============================
# 1) summary plot
# 어떤 변수가 중요한지
# ==============================
plt.figure()
shap.summary_plot(shap_values, X_shap_sample, show=False)
plt.title("Type1 Stage1 SHAP Summary Plot")
plt.tight_layout()
plt.show()

# ==============================
# 2) bar plot
# 평균 절대 SHAP 기준 중요 변수
# ==============================
plt.figure()
shap.summary_plot(shap_values, X_shap_sample, plot_type="bar", show=False)
plt.title("Type1 Stage1 SHAP Feature Importance")
plt.tight_layout()
plt.show()

# ==============================
# 3) 중요 변수 Top10 표로 확인
# ==============================
mean_abs_shap = pd.Series(
    abs(shap_values).mean(axis=0),
    index=X_shap_sample.columns
).sort_values(ascending=False)

print("\n[Type1 Stage1 SHAP 중요 변수 TOP 10]")
display(mean_abs_shap.head(10).to_frame("mean_|SHAP|"))

In [ ]:
# ==============================
# 불균형 가중치 계산
# ==============================

neg = (y_train_stage1 == 0).sum()
pos = (y_train_stage1 == 1).sum()

base_weight = neg / pos
print("base_weight:", base_weight)

scale_weight = neg / pos

print("정상 개수:", neg)
print("불량 개수:", pos)
print("scale_pos_weight:", scale_weight)

precision	모델이 불량이라고 예측한 것 중 실제 불량 비율

recall	실제 불량 중에서 모델이 잡아낸 비율

f1-score	precision과 recall의 균형 점수

support	실제 데이터 개수

In [ ]:
results_df.head(10)

In [ ]:
# ==============================
# Stage1 최종 예측 저장
# ==============================
y_prob_train_best = best_model.predict_proba(X_train)[:, 1]
y_pred_train_stage1 = (y_prob_train_best >= best_threshold).astype(int)

y_prob_test_best = best_model.predict_proba(X_test)[:, 1]
y_pred_stage1 = (y_prob_test_best >= best_threshold).astype(int)

print(f"\n=== Stage 1 최종 결과 (best threshold={best_threshold}) ===")
print(classification_report(y_test_stage1, y_pred_stage1, zero_division=0))

정밀도 (Precision): 진짜 불량인 비율  100
정상을 정상이라고 맞춘 확률 96%   
불량을 불량이라고 맞춘 확률 29%

재현율 (Recall): 실제 불량 중 찾아낸 비율 (불량 탐지율)
정상 
불량 

행이 0이면 → 정상(0)을 얼마나 잘 맞췄는지 / 정상을 
행이 1이면 → 불량(1)을 얼마나 잘 맞췄는지 / 

In [ ]:
# 구조 + 표면 동시에 발생한 데이터
both_defect = (y_train["구조"] == 1) & (y_train["표면"] == 1)

print("구조 + 표면 중복 불량 개수:", both_defect.sum())

print("전체 데이터 대비 비율:")
print(both_defect.mean())

#test 데이터 확인
both_defect_test = (y_test["구조"] == 1) & (y_test["표면"] == 1)

print("test 중복 불량:", both_defect_test.sum())

In [ ]:
# ==============================
# Stage2용 라벨 생성 함수
# 정상 / 구조 / 표면
# ==============================
def make_stage2_label(row):
    if row["구조"] == 1:
        return "구조"
    elif row["표면"] == 1:
        return "표면"
    else:
        return "정상"

In [ ]:
# ==============================
# Stage2 학습 데이터 (train)
# ==============================
stage2_train_idx = (y_pred_train_stage1 == 1)

X_train_stage2 = X_train[stage2_train_idx].copy()
y_train_stage2 = y_train[stage2_train_idx].copy()

y_train_stage2_cls = y_train_stage2.apply(make_stage2_label, axis=1)

print("[Stage2 train class 분포]")
print(y_train_stage2_cls.value_counts())


# ==============================
# Stage2 테스트 데이터 (test)
# ==============================
stage2_test_idx = (y_pred_stage1 == 1)

X_test_stage2 = X_test[stage2_test_idx].copy()
y_test_stage2 = y_test[stage2_test_idx].copy()

y_test_stage2_cls = y_test_stage2.apply(make_stage2_label, axis=1)

print("\n[Stage2 test class 분포]")
print(y_test_stage2_cls.value_counts())

In [ ]:
# ==============================
# 문자열 라벨 -> 숫자 라벨
# ==============================
label_map = {
    "정상": 0,
    "구조": 1,
    "표면": 2
}

inv_label_map = {v: k for k, v in label_map.items()}

y_train_stage2_enc = y_train_stage2_cls.map(label_map)
y_test_stage2_enc = y_test_stage2_cls.map(label_map)

In [ ]:
# ==============================
# Stage2 클래스 가중치
# ==============================
class_counts = y_train_stage2_enc.value_counts().sort_index()
n_classes = len(class_counts)
n_samples = len(y_train_stage2_enc)

class_weight_dict = {
    cls: n_samples / (n_classes * count)
    for cls, count in class_counts.items()
}

sample_weight_stage2 = y_train_stage2_enc.map(class_weight_dict)

print("\n[Stage2 class weight]")
for cls_num, weight in class_weight_dict.items():
    print(f"{inv_label_map[cls_num]}: {weight:.4f}")

[중요] 무엇을 가중치를 어떻게 매긴건지

In [ ]:
# ==============================
# Stage2 모델 (정상 / 구조 / 표면)
# ==============================
model_stage2 = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=-1
)

model_stage2.fit(
    X_train_stage2,
    y_train_stage2_enc,
    sample_weight=sample_weight_stage2
)

y_pred_stage2_enc = model_stage2.predict(X_test_stage2)

# 숫자 -> 한글 라벨 복원
y_pred_stage2 = pd.Series(y_pred_stage2_enc, index=y_test_stage2_enc.index).map(inv_label_map)
y_true_stage2 = y_test_stage2_enc.map(inv_label_map)

print("\n=== Stage 2: 정상 / 구조 / 표면 분류 ===")
print(classification_report(y_true_stage2, y_pred_stage2, zero_division=0))

In [ ]:
import shap
import numpy as np
import pandas as pd

# ==============================
# SHAP 분석용 샘플
# ==============================
X_sample_stage2 = X_test_stage2.sample(
    min(200, len(X_test_stage2)),
    random_state=42
).copy()

# 컬럼명 문자열 변환 (MultiIndex 방지)
X_sample_stage2.columns = [str(c) for c in X_sample_stage2.columns]

# ==============================
# SHAP 계산
# ==============================
explainer_stage2 = shap.TreeExplainer(model_stage2)
shap_values_stage2 = explainer_stage2.shap_values(X_sample_stage2)

In [ ]:
class_names = ["정상", "구조", "표면"]

if isinstance(shap_values_stage2, list):

    shap_importance_normal = pd.Series(
        np.abs(shap_values_stage2[0]).mean(axis=0),
        index=X_sample_stage2.columns
    ).sort_values(ascending=False)

    shap_importance_structure = pd.Series(
        np.abs(shap_values_stage2[1]).mean(axis=0),
        index=X_sample_stage2.columns
    ).sort_values(ascending=False)

    shap_importance_surface = pd.Series(
        np.abs(shap_values_stage2[2]).mean(axis=0),
        index=X_sample_stage2.columns
    ).sort_values(ascending=False)

else:
    shap_arr = np.array(shap_values_stage2)

    shap_importance_normal = pd.Series(
        np.abs(shap_arr[:, :, 0]).mean(axis=0),
        index=X_sample_stage2.columns
    ).sort_values(ascending=False)

    shap_importance_structure = pd.Series(
        np.abs(shap_arr[:, :, 1]).mean(axis=0),
        index=X_sample_stage2.columns
    ).sort_values(ascending=False)

    shap_importance_surface = pd.Series(
        np.abs(shap_arr[:, :, 2]).mean(axis=0),
        index=X_sample_stage2.columns
    ).sort_values(ascending=False)

In [ ]:
# ========================================
# 시각화: 표면 vs 구조 중요도 비교
# ========================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_n = 15

# 표면 불량
shap_importance_surface.head(top_n)[::-1].plot(
    kind='barh', ax=axes[0], color='steelblue'
)
axes[0].set_title("Type1 표면 불량 - SHAP Feature Importance", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Mean |SHAP value|")
axes[0].grid(axis='x', alpha=0.3)

# 구조 불량
shap_importance_structure.head(top_n)[::-1].plot(
    kind='barh', ax=axes[1], color='coral'
)
axes[1].set_title("Type1 구조 불량 - SHAP Feature Importance", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Mean |SHAP value|")
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# shap_values_stage2가 list 형태인 경우
if isinstance(shap_values_stage2, list):
    shap_surface = shap_values_stage2[2]     # 표면
    shap_structure = shap_values_stage2[1]   # 구조

# shap_values_stage2가 array 형태인 경우
else:
    shap_arr = np.array(shap_values_stage2)
    shap_surface = shap_arr[:, :, 2]         # 표면
    shap_structure = shap_arr[:, :, 1]       # 구조

# 표면 불량 summary plot
print("\n=== Type1 표면 불량 SHAP Summary Plot ===")
shap.summary_plot(shap_surface, X_sample_stage2)

# 구조 불량 summary plot
print("\n=== Type1 구조 불량 SHAP Summary Plot ===")
shap.summary_plot(shap_structure, X_sample_stage2)